In [ ]:
# =============================================================================
# 1. Setup
# Ocean heat content (OHC) forward emulator for ACCESS-OM2.
#
# This notebook only holds the experiment settings (cell 2) and the calls that
# run it. The data pipeline, model, losses, training loop and diagnostics live
# in src/Experiment/ (built on src/Emulator/ for the UNet and the loss terms).
#
# Cells: 1 setup | 2 config | 3 data | 4 known-closure check | 5 model + losses
#        6 training | 7 skill test | 8 control run | 9 plots
# =============================================================================
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Path to the OM2-emulator repository. Leave as None to find it automatically;
# set it (e.g. "/home/156/txs156/uom/OM2-emulator") if that fails.
REPO_ROOT = None

# Places to look for src/Experiment, in order: REPO_ROOT, the OM2_EMULATOR_REPO
# environment variable, the notebook file's folder (VS Code provides it as
# __vsc_ipynb_file__), then the working directory. Each is searched upwards,
# because the kernel's working directory is not always the notebook's folder.
_starts = [
    Path(p).expanduser()
    for p in (REPO_ROOT, os.environ.get("OM2_EMULATOR_REPO"), globals().get("__vsc_ipynb_file__"), Path.cwd())
    if p
]
_candidates = [q for p in _starts for q in (p, *p.parents)]
SRC = next((q / "src" for q in _candidates if (q / "src" / "Experiment").is_dir()), None)
if SRC is None:
    raise FileNotFoundError(
        f"Could not find src/Experiment. Working directory: {Path.cwd()}. Looked in and above: "
        f"{[str(p) for p in _starts]}. Set REPO_ROOT above to the OM2-emulator repository."
    )
sys.path.insert(0, str(SRC))

from Experiment import (
    DataConfig, EvalConfig, ExperimentConfig, LossConfig, ModelConfig, NormalisationConfig,
    TimeConfig, TrainConfig, WindowConfig,
    build_data, build_losses, build_model, build_module, build_trainer,
    check_known_closure, count_parameters, run_control, run_skill_test,
    plot_control, plot_global_rmse, plot_global_timeseries, plot_snapshots, plot_training_history,
)

print(f"src: {SRC}")
print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
# =============================================================================
# 2. Experiment configuration -- every setting of the run lives here.
# Dates are "YYYY-MM" and all date ranges are inclusive. The config is checked
# for consistency as soon as it is created.
# =============================================================================
cfg = ExperimentConfig(
    # ---- Variables and normalisation ---------------------------------------
    data=DataConfig(
        path="/g/data/nm47/txs156/OM2-emulator/data/1deg_ocean_heat_emulator_data.nc",
        # Prognostic = predicted AND fed back as inputs at the next step (the outputs).
        prognostic=["ocean_heat_content_2d"],
        # Forcing = prescribed from the data at every step. Drop tau_x/tau_y for
        # heat-flux-only forcing.
        forcing=["total_surface_heat_flx", "tau_x", "tau_y"],
        normalisation=NormalisationConfig(
            # Spatial_climatology | Spatial_climatology_global_variance | Spatial_in_time | Global_in_time
            strategy="Spatial_climatology",
            fit_period=("1970-01", "2004-02"),  # months used to fit mean/std (inside the training period)
        ),
        cache_dir=None,  # None -> <data dir>/emulator_cache; rebuilt automatically when data settings change
    ),
    # ---- Time periods --------------------------------------------------------
    # train/valid are the INITIAL months of the samples (the last prior state);
    # each sample also needs n_prior - 1 earlier and posterior_steps later months.
    time=TimeConfig(
        time_axis=("1970-01", "2018-12"),
        train=("1970-02", "2004-01"),
        valid=("2004-02", "2005-01"),
        test=("2005-02", "2018-11"),     # skill test: seeded at 2005-02, predicts 2005-03 .. 2018-12
        control=("2003-01", "2003-12"),  # control run: this year's forcing repeated (near-zero net flux)
        control_repeats=10,
    ),
    # ---- Rollout windows -----------------------------------------------------
    window=WindowConfig(
        n_prior=2,             # past states the model sees
        posterior_steps=12,    # targets available per sample
        rollout_steps=12,      # autoregressive steps per training sample
        rollout_schedule=None, # e.g. {0: 1, 20: 4, 50: 12}: start short, cost scales with steps
        valid_rollout_steps=None,  # None -> rollout_steps
    ),
    # ---- Loss terms and weights ----------------------------------------------
    loss=LossConfig(
        # local_mse | spectral | global_closure. 0 switches a term off. A weight
        # can also be a list of absolute per-step weights (length posterior_steps).
        terms={"local_mse": 1.0, "spectral": 0.0, "global_closure": 0.1},
        step_weights=None,     # relative emphasis per lead time, e.g. [1] * 6 + [2] * 6 (rescaled to mean 1)
        surface_flux_sign=1.0, # +1: positive heat flux warms the ocean
        closure_min_scale=1.0e20,
    ),
    # ---- Architecture ----------------------------------------------------------
    model=ModelConfig(
        arch="unet",
        padding_mode="zeros",  # ~2x faster than "replicate", ~35% less GPU memory
        # "spatial_residual" = the former ResNet notebook: a learned full-resolution
        # correction of the prediction. Adds memory: lower batch_size if needed.
        output_head=None,
        output_head_hidden_channels=16,
        output_head_residual_scale=1.0,
        latent_processor=None,  # or "latent_residual" (refines the UNet bottleneck)
    ),
    # ---- Training ---------------------------------------------------------------
    train=TrainConfig(
        batch_size=64,
        accumulate_grad_batches=1,  # effective batch = batch_size * this
        max_epochs=200,
        lr=1e-4,
        weight_decay=0.0,
        lr_scheduler=None,          # or "cosine"
        gradient_clip_val=None,
        precision="32-true",        # or "16-mixed" (~1.5x faster on a V100)
        seed=42,
        run_dir=None,               # e.g. "../runs/unet_baseline": saves config, CSV log, checkpoints
    ),
    # ---- Evaluation -------------------------------------------------------------
    eval=EvalConfig(
        run_skill_test=True,
        run_control=False,
        snapshot_periods=["2005-05", "2010-12", "2015-12"],  # months "YYYY-MM" or year means "YYYY"
        anomaly_scale=2e9,
        difference_scale=2e9,
    ),
)

In [ ]:
# =============================================================================
# 3. Data: normalise every variable once over the whole time axis (cached on
# disk, keyed by the data settings), then serve rollout windows from it.
# =============================================================================
data = build_data(cfg)

In [ ]:
# =============================================================================
# 4. Known-closure check: the closure loss on the TRUE data, i.e. the lowest
# value the global_closure term can reach. See issue #43: with monthly means
# the anomaly budget scores ~0.5, so a perfect model is still penalised.
# =============================================================================
known_closure = None
if cfg.loss.terms.get("global_closure", 0):
    known_closure = check_known_closure(cfg, data)

In [ ]:
# =============================================================================
# 5. Model, loss terms and the Lightning module.
# =============================================================================
model = build_model(cfg, data)
losses = build_losses(cfg, data)
module = build_module(cfg, model, losses, data)

print(
    f"Model: {cfg.model.arch} | output_head={cfg.model.output_head} | "
    f"latent_processor={cfg.model.latent_processor} | {count_parameters(model):,} parameters"
)
for term in losses:
    print(f"  {term.name:15s} step weights {term.step_weights}")

In [ ]:
%%time
# =============================================================================
# 6. Training. Check nvidia-smi first: other kernels on the same GPU slow
# training down and can cause CUDA out-of-memory errors (build_trainer warns).
# =============================================================================
trainer = build_trainer(cfg)
trainer.fit(module, data.train_dl, data.valid_dl)

In [ ]:
# =============================================================================
# 7. Skill test: one free-running rollout from the truth at time.test[0],
# driven by the real (held-out) forcing. Returns predictions and truth in
# physical units, as full fields and as anomalies.
# =============================================================================
skill = run_skill_test(cfg, model, data) if cfg.eval.run_skill_test else None
skill

In [ ]:
# =============================================================================
# 8. Control run: the forcing of time.control repeated time.control_repeats
# times, to isolate drift under near-constant forcing.
# =============================================================================
control = run_control(cfg, model, data) if cfg.eval.run_control else None

In [ ]:
# =============================================================================
# 9. Plots
# =============================================================================
if skill is not None:
    plot_snapshots(skill, cfg.eval.snapshot_periods, cfg.eval.anomaly_scale, cfg.eval.difference_scale)
    plot_global_timeseries(skill)
    plot_global_rmse(skill)
if control is not None:
    plot_control(control)
if cfg.train.run_dir is not None:
    plot_training_history(cfg.train.run_dir)
plt.show()